In [1]:
# ============================================================
# Imports
# ============================================================

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# Load Dataset
# ============================================================

DATA_PATH = (
    "data/"
    "marketing_sample_for_amazon_com-amazon_fashion_products__20200201_20200430__30k_data.ldjson"
)

df = pd.read_json(
    DATA_PATH,
    lines=True
)

print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (30000, 33)


,uniq_id,crawl_timestamp,asin,product_url,product_name,image_urls__small,medium,large,browsenode,brand,...,colour,no__of_reviews,seller_name,seller_id,left_in_stock,no__of_offers,no__of_sellers,technical_details__k_v_pairs,formats___editions,name_of_author_for_books
0,26d41bdc1495de290bc8e6062d927729,2020-02-07 05:11:36 +0000,B07STS2W9T,https://www.amazon.in/Facon-Kalamkari-Handbloc...,LA' Facon Cotton Kalamkari Handblock Saree Blo...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,1.968255e+09,LA' Facon,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,410c62298852e68f34c35560f2311e5a,2020-02-07 08:45:56 +0000,B07N6TD2WL,https://www.amazon.in/Sf-Jeans-Pantaloons-T-Sh...,Sf Jeans By Pantaloons Men's Plain Slim fit T-...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,1.968123e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,52e31bb31680b0ec73de0d781a23cc0a,2020-02-06 11:09:38 +0000,B07WJ6WPN1,https://www.amazon.in/LOVISTA-Traditional-Prin...,LOVISTA Cotton Gota Patti Tassel Traditional P...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,1.968255e+09,LOVISTA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25798d6dc43239c118452d1bee0fb088,2020-02-07 08:32:45 +0000,B07PYSF4WZ,https://www.amazon.in/People-Printed-Regular-T...,People Men's Printed Regular fit T-Shirt,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,1.968123e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ad8a5a196d515ef09dfdaf082bdc37c4,2020-02-06 14:27:48 +0000,B082KXNM7X,https://www.amazon.in/Monte-Carlo-Cotton-Colla...,Monte Carlo Grey Solid Cotton Blend Polo Colla...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,1.968070e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# ============================================================
# Dataset Overview
# ============================================================

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.info()

Rows: 30000
Columns: 33
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 33 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   uniq_id                        30000 non-null  object 
 1   crawl_timestamp                30000 non-null  object 
 2   asin                           30000 non-null  object 
 3   product_url                    30000 non-null  object 
 4   product_name                   30000 non-null  object 
 5   image_urls__small              29998 non-null  object 
 6   medium                         29998 non-null  object 
 7   large                          28841 non-null  object 
 8   browsenode                     29480 non-null  float64
 9   brand                          21857 non-null  object 
 10  sales_price                    27110 non-null  float64
 11  weight                         30000 non-null  object 
 12  rating                

In [4]:
# ============================================================
# Missing Value Analysis
# ============================================================

missing_df = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

missing_df.columns = [
    "column",
    "missing_count"
]

missing_df["missing_percentage"] = (
    missing_df["missing_count"]
    / len(df)
    * 100
)

missing_df.head(20)

,column,missing_count,missing_percentage
0,name_of_author_for_books,29999,99.996667
1,formats___editions,29998,99.993333
2,no__of_sellers,28980,96.600000
3,no__of_offers,28980,96.600000
4,technical_details__k_v_pairs,28846,96.153333
5,left_in_stock,26943,89.810000
6,no__of_reviews,26548,88.493333
7,colour,23971,79.903333
8,seller_id,21636,72.120000
9,seller_name,21636,72.120000


In [5]:
# ============================================================
# Numerical Feature Analysis
# ============================================================

numeric_cols = [
    "sales_price",
    "rating"
]

df[numeric_cols].describe()

,sales_price,rating
count,27110.000000,30000.000000
mean,862.172397,4.039857
std,964.223008,0.840009
min,39.000000,1.000000
25%,379.000000,3.500000
50%,590.000000,4.000000
75%,899.000000,4.900000
max,9988.000000,5.000000


In [6]:
# ============================================================
# Brand Analysis
# ============================================================

print(
    "Unique Brands:",
    df["brand"].nunique()
)

df["brand"].value_counts().head(20)

Unique Brands: 6458


brand
Max                       504
Generic                   245
BIBA                      205
Mothercare                156
Campus Sutra              150
Soch                      149
nauti nati                132
Ada                       127
GRITSTONES                110
PrintOctopus              102
Allen Solly Junior         93
Columbia                   92
Indian Terrain             80
Excalibur by Unlimited     80
Buckle Down                78
ANNI DESIGNER              78
MIMOSA                     76
Gini & Jony                69
Arrow Sports               68
Harpa                      66
Name: count, dtype: int64

In [7]:
# ============================================================
# Product Text Analysis
# ============================================================

df["combined_text"] = (
    df["product_name"]
        .fillna("")
    + " "
    + df["meta_keywords"]
        .fillna("")
)

df["combined_text_length"] = (
    df["combined_text"]
        .str.len()
)

df["combined_text_length"].describe()

count    30000.00000
mean       136.84650
std         63.23016
min         14.00000
25%         95.00000
50%        125.00000
75%        164.00000
max       1015.00000
Name: combined_text_length, dtype: float64

In [8]:
# ============================================================
# Weight Analysis
# ============================================================

print(
    df["weight"]
    .value_counts()
    .head(20)
)

weight
999999999    23745
200 g          736
249 g          689
299 g          398
399 g          304
349 g          259
99.8 g         251
449 g          228
150 g          190
499 g          168
181 g          163
272 g          117
68 g           115
113 g          113
90.7 g         108
222 g           97
86.2 g          94
227 g           80
59 g            80
281 g           66
Name: count, dtype: int64


In [9]:
df["weight"].sample(20)

12175    999999999
15296        272 g
19088        399 g
9513         372 g
12289    999999999
21801    999999999
21757    999999999
10048    999999999
9102         249 g
16342    999999999
28761        249 g
7203     999999999
2506     999999999
11659    999999999
24545    999999999
14899    999999999
27299    999999999
16676        299 g
25028    999999999
9452         200 g
Name: weight, dtype: object

In [25]:
def prepare_data(self):

    # --------------------------------------------------------
    # Missing value handling
    # --------------------------------------------------------

    self.df["brand"] = (
        self.df["brand"]
        .fillna("unknown")
    )

    self.df["sales_price"] = (
        self.df["sales_price"]
        .fillna(
            self.df["sales_price"].median()
        )
    )

    # --------------------------------------------------------
    # Category text
    # --------------------------------------------------------

    self.df["category_text"] = (
        self.df["parent___child_category__all"]
        .fillna("")
        .astype(str)
    )

    # --------------------------------------------------------
    # Combined text for similarity
    # --------------------------------------------------------

    self.df["combined_text"] = (
        self.df["product_name"].fillna("")
        + " "
        + self.df["meta_keywords"].fillna("")
        + " "
        + self.df["category_text"]
    )

    return self

In [26]:
ProductSimilarityEngine.prepare_data = prepare_data

In [27]:
def build_features(self):

    # --------------------------------------------------------
    # Text Features (Most Important Signal)
    # --------------------------------------------------------

    self.vectorizer = TfidfVectorizer(
        max_features=10000,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2
    )

    text_features = (
        self.vectorizer.fit_transform(
            self.df["combined_text"]
        )
    )

    # --------------------------------------------------------
    # Brand Features
    # --------------------------------------------------------

    self.brand_encoder = OneHotEncoder(
        handle_unknown="ignore"
    )

    brand_features = (
        self.brand_encoder.fit_transform(
            self.df[["brand"]]
        )
    )

    # --------------------------------------------------------
    # Numerical Features
    # --------------------------------------------------------

    self.scaler = StandardScaler()

    numeric_features = (
        self.scaler.fit_transform(
            self.df[
                ["sales_price", "rating"]
            ]
        )
    )

    numeric_features = csr_matrix(
        numeric_features
    )

    # --------------------------------------------------------
    # Hybrid Feature Matrix
    # --------------------------------------------------------
    # Text   : 85%
    # Brand  : 10%
    # Numeric:  5%
    # --------------------------------------------------------

    self.feature_matrix = (
        hstack([
            text_features * 0.85,
            brand_features * 0.10,
            numeric_features * 0.05
        ])
        .tocsr()
    )

    return self

In [28]:
ProductSimilarityEngine.build_features = build_features

In [29]:
engine = (
    ProductSimilarityEngine(df)
    .prepare_data()
    .build_features()
    .build_index()
)

print("Engine Rebuilt Successfully")

Engine Rebuilt Successfully


In [30]:
engine.find_similar_products_with_scores(
    df.iloc[0]["uniq_id"],
    10
)

,uniq_id,product_name,brand,sales_price,rating,similarity_score
18915,4188c32471f46876ed222dfcae360791,LA' Facon Cotton Kalamkari Handblock Saree Blo...,LA' Facon,200.0,5.0,0.940535
10141,2d249ce6637640fe9ef8a5cf20cbc9dc,Cotton Kalamkari Handblock Saree Blouse/Kurti ...,La' Facon,200.0,3.0,0.594362
5360,7d5bb9bfd4df44033f49993750023708,Phoenix Retail's Kalamkari Maysore Silk Saree ...,Generic,449.0,4.0,0.274967
315,52d36de6bca1e55d21b8a646f575fe05,Looks Akira Cotton with Blouse Piece Saree (LA...,Looks Akira,399.0,2.5,0.265900
9943,2f0a08c8caedbe824a1135bbc042eb56,Aaradhya Fashion Women's Bhagalpuri Kalamkari ...,Aaradhya Fashion,449.0,4.0,0.239532
19612,2273dc9330251b629313b0830ceac78b,HANDBLOCK PRINT Women's Rayon Embroidered Anar...,HANDBLOCK PRINT,543.0,4.2,0.235638
23115,90657dce8ca4b3819a4577ead53bbb55,SareeShop Indian Women's Kalamkari Silk Saree ...,SareeShop,1455.0,4.0,0.227834
25000,29d60d0b307ccda1b20eaaf6c83f1471,Handblock Print Women’s Rayon Gotta Work Strai...,Handblock Print,589.0,5.0,0.227074
27920,e264323af79b3c445c96375870623fb3,Velmita Women's Cotton Silk Saree With Blouse ...,Velmita,539.0,4.0,0.226799
21819,84f1b19f28adf37c93a5b36fb84bffcf,La Blanca Women's Deco Stud Halter Mio One Pie...,La Blanca,590.0,3.7,0.213710
